In [4]:
import xarray as xr
import numpy as np
import pandas as pd
import os
import sys
import argparse

In [5]:
root_dir = "/pscratch/sd/b/beharrop/kmscale_hackathon/hackathon_pre/"
in_file = f"{root_dir}screamv2_ne120_tracking/screamv2_ne120_hp8.etc_stitched_nodes.txt"
in_file

'/pscratch/sd/b/beharrop/kmscale_hackathon/hackathon_pre/screamv2_ne120_tracking/screamv2_ne120_hp8.etc_stitched_nodes.txt'

In [6]:
mcs_dir = "/pscratch/sd/w/wcmca1/hackathon/mcs/scream/stats/"
mcs_trackfile = f"{mcs_dir}/mcs_tracks_final_20190801.0000_20200901.0000.nc"
mcs_trackfile

'/pscratch/sd/w/wcmca1/hackathon/mcs/scream/stats//mcs_tracks_final_20190801.0000_20200901.0000.nc'

In [7]:
# Parse the storm data text file
def parse_storm_file(file_path, unstructured_mesh=True):
    storm_data = []
    with open(file_path, 'r') as f:
        storm_id = 0
        for line in f:
            line = line.strip()
            if line.startswith("start"):
                # New storm
                storm_id += 1
                num_timesteps, year, month, day, hour = map(int, line.split()[1:])
            else:
                # Storm details
                cols = line.split()
                if unstructured_mesh:
                    storm_data.append({
                        "storm_id": storm_id,
                        "grid_id": int(cols[0]),
                        "lon": float(cols[1]),
                        "lat": float(cols[2]),
                        "year": int(cols[-4]),
                        "month": int(cols[-3]),
                        "day": int(cols[-2]),
                        "hour": int(cols[-1])
                    })
                else:
                    storm_data.append({
                        "storm_id": storm_id,
                        "lon_id": int(cols[0]),
                        "lat_id": int(cols[1]),
                        "lon": float(cols[2]),
                        "lat": float(cols[3]),
                        "year": int(cols[-4]),
                        "month": int(cols[-3]),
                        "day": int(cols[-2]),
                        "hour": int(cols[-1])
                    })
    return pd.DataFrame(storm_data)

In [11]:
unstructured_mesh = True
storm_df = parse_storm_file(in_file, unstructured_mesh=unstructured_mesh)
storm_df

,storm_id,grid_id,lon,lat,year,month,day,hour
0,1,225172,328.349301,51.062119,2019,8,1,0
1,1,225144,330.365876,51.836503,2019,8,1,6
2,1,227884,332.185954,52.994706,2019,8,1,12
3,1,227986,332.396932,53.956885,2019,8,1,18
4,1,249882,330.896761,55.873462,2019,8,2,0
...,...,...,...,...,...,...,...,...
18699,1329,663199,237.147256,-59.867208,2020,8,30,18
18700,1329,662752,243.120830,-62.508739,2020,8,31,0
18701,1329,661227,249.306602,-64.761070,2020,8,31,6
18702,1329,661044,255.000040,-67.376398,2020,8,31,12


In [ ]:
# ETC 1014: north Pacific, 2020-05-18 18Z - 2020-05-24 18Z
# 3-way overlap period: 2020-05-15 18Z - 2020-05-22 12Z

# np.where(storm_df['storm_id'] == 1014)
storm_df.loc[storm_df['storm_id'] == 1014]

,storm_id,grid_id,lon,lat,year,month,day,hour
14270,1014,167911,202.148438,40.424109,2020,5,20,18
14271,1014,170650,204.061587,41.810446,2020,5,21,0
14272,1014,170706,205.379970,43.008680,2020,5,21,6
14273,1014,170812,207.222196,44.399883,2020,5,21,12
14274,1014,171619,209.799978,47.945558,2020,5,21,18
14275,1014,171909,210.353753,50.480121,2020,5,22,0
14276,1014,171982,210.519781,52.416100,2020,5,22,6
14277,1014,172027,209.769209,53.764658,2020,5,22,12
14278,1014,177491,209.145055,54.149000,2020,5,22,18
14279,1014,177495,209.450240,54.532924,2020,5,23,0


In [9]:
mcs_trackstats = xr.open_dataset(mcs_trackfile)

# ONLY load essential variables
required_vars = ['meanlon', 'meanlat', 'base_time']
subset = mcs_trackstats[required_vars].compute()
df_all = subset.to_dataframe().reset_index()
df_all
# print(f"Loaded {len(df_all)} total track-time points")

# # Filter by tracks from land fraction file if provided
# if tracks_to_process is not None:
#     df_all = df_all[df_all['tracks'].isin(tracks_to_process)]
#     print(f"Filtered to {len(df_all)} track-time points from land fraction file")

,tracks,times,meanlon,meanlat,base_time
0,0,0,-146.117874,-40.489239,2019-08-01 01:00:00
1,0,1,-148.803543,-43.098492,2019-08-01 02:00:00
2,0,2,-148.724335,-43.480473,2019-08-01 03:00:00
3,0,3,-149.336044,-44.175705,2019-08-01 04:00:00
4,0,4,-150.073959,-44.962494,2019-08-01 05:00:00
...,...,...,...,...,...
36716545,56486,645,NaN,NaN,NaT
36716546,56486,646,NaN,NaN,NaT
36716547,56486,647,NaN,NaN,NaT
36716548,56486,648,NaN,NaN,NaT


In [10]:
df_all['base_time']

0          2019-08-01 01:00:00
1          2019-08-01 02:00:00
2          2019-08-01 03:00:00
3          2019-08-01 04:00:00
4          2019-08-01 05:00:00
                   ...        
36716545                   NaT
36716546                   NaT
36716547                   NaT
36716548                   NaT
36716549                   NaT
Name: base_time, Length: 36716550, dtype: datetime64[ns]